In [10]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers


In [11]:
df = pd.read_csv(
    "D:/xampp/htdocs/kuafor_proje/kahramanmaras_yorumlar.csv",
    sep=";",
    encoding="utf-8"
)

df.columns = df.columns.str.strip().str.lower()
df = df[["yorum", "yorum puani"]].copy()

def temizle(t):
    t = str(t).lower()
    t = re.sub(r"[^\w\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text"] = df["yorum"].apply(temizle)

# Genel duygu etiketi
df["sentiment"] = df["yorum puani"].apply(lambda x: 1 if x >= 4 else 0)

df = df[df["text"].str.len() > 0].copy()


In [12]:
ASPECTS = {
    "kesim": ["saç", "kesim", "sakal", "fön", "boya", "tırnak"],
    "bekleme": ["bekledim", "bekleme", "sıra", "randevu", "gecikti", "uzun"],
    "fiyat": ["fiyat", "pahalı", "ucuz", "ücret", "para"],
    "hijyen": ["temiz", "kirli", "hijyen", "pis"],
    "personel": ["personel", "çalışan", "usta", "ilgili", "kaba"]
}

NEG_OVERRIDE = [
    "kötü", "berbat", "rezalet", "iğrenç",
    "beğenmedim", "memnun değil", "pişman", "vasat"
]


In [13]:
MAX_WORDS = 20000
MAX_LEN = 40

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

X = tokenizer.texts_to_sequences(df["text"])
X = pad_sequences(X, maxlen=MAX_LEN, padding="post")

y = df["sentiment"].values


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [15]:
model = tf.keras.Sequential([
    layers.Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 40, 128)           2560000   
                                                                 
 bidirectional (Bidirection  (None, 128)               98816     
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 2658945 (10.14 MB)
Trainable params: 2658945 (10.14 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32
)


Epoch 1/5
17/17 [==============================] - 4s 92ms/step - loss: 0.5080 - accuracy: 0.8194 - val_loss: 0.5588 - val_accuracy: 0.7704
Epoch 2/5
17/17 [==============================] - 1s 44ms/step - loss: 0.3499 - accuracy: 0.8380 - val_loss: 0.4755 - val_accuracy: 0.7704
Epoch 3/5
17/17 [==============================] - 1s 48ms/step - loss: 0.2428 - accuracy: 0.9181 - val_loss: 0.3942 - val_accuracy: 0.8667
Epoch 4/5
17/17 [==============================] - 1s 45ms/step - loss: 0.0944 - accuracy: 0.9758 - val_loss: 0.2596 - val_accuracy: 0.8815
Epoch 5/5
17/17 [==============================] - 1s 63ms/step - loss: 0.0498 - accuracy: 0.9851 - val_loss: 0.3381 - val_accuracy: 0.8667


In [17]:
pred_prob = model.predict(X_test).reshape(-1)
pred = (pred_prob >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1-score :", f1_score(y_test, pred))


6/6 [==============================] - 1s 8ms/step
Accuracy : 0.8875739644970414
Precision: 0.879746835443038
Recall   : 1.0
F1-score : 0.936026936026936


In [18]:
def extract_aspects(text):
    found = []
    for asp, kws in ASPECTS.items():
        if any(k in text for k in kws):
            found.append(asp)
    return found


def predict_hybrid(text):
    t = temizle(text)

    seq = tokenizer.texts_to_sequences([t])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post")

    prob = float(model.predict(pad, verbose=0)[0][0])

    # NEGATIVE OVERRIDE
    if any(w in t for w in NEG_OVERRIDE):
        sentiment = "Olumsuz"
        prob = min(prob, 0.2)
    else:
        sentiment = "Olumlu" if prob >= 0.5 else "Olumsuz"

    return {
        "yorum": text,
        "sentiment": sentiment,
        "sentiment_score": prob,
        "aspects": extract_aspects(t)
    }


In [21]:
predict_hybrid("Saç kesimi kötü.")


{'yorum': 'Saç kesimi kötü.',
 'sentiment': 'Olumsuz',
 'sentiment_score': 0.2,
 'aspects': ['kesim']}